# CartPole 

This is a chaotic problem with no stable equlibrium.  The goal is to balance the pole, and the game ends with the pole irretrievably falls over or the cart goes outside of the allowed boundaries.  Surviving 1000 timesteps is considered "solved".

At each timestep the agent sees the 4-number state and outputs a probability distribution over actions (push left or push right).  The REINFORCE algorithm updates the policy to make actions that led to high returns more probable.


In [ ]:
import subprocess
import sys
import platform

subprocess.run([sys.executable, '-m', 'pip', 'install',
                'gymnasium[classic-control]', 'moviepy', '-q'], check=True)

# Virtual display only needed on Linux (Colab) — not on Windows or Mac
if platform.system() == 'Linux':
    subprocess.run(['apt-get', 'install', '-y', 'xvfb', '-q'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'pyvirtualdisplay', '-q'], check=True)
    from pyvirtualdisplay import Display
    display = Display(visible=0, size=(600, 400))
    display.start()
    print("Virtual display started (Linux/Colab mode).")
else:
    print(f"Running on {platform.system()} — no virtual display needed.")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical
import gymnasium as gym
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from collections import deque

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# --- Hyperparameters ---
GAMMA          = 0.99    # Discount factor
LR             = 3e-4
HIDDEN         = 128
MAX_EPISODES   = 2000
SOLVED_SCORE   = 800     # Average over 50 episodes
RENDER_EVERY   = 50     # Render a video every N episodes so we can watch progress
RENDER_STEPS   = 1000    

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")


## 1. The Policy Network

The policy is a small MLP that maps the 4-dimensional state to a probability distribution over the 2 actions. The policy gradient network does not require a value function, target network, or replay buffer. Just a network that outputs action probabilities, and a training signal derived from actual episode returns.


In [ ]:
class PolicyNet(nn.Module):
    """
    Maps state -> action probabilities.
    Two hidden layers with ReLU; softmax output over actions.
    """
    def __init__(self, state_dim, action_dim, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )

    def forward(self, x):
        return torch.softmax(self.net(x), dim=-1)

    def act(self, state):
        """Sample an action and return it with its log probability."""
        state  = torch.tensor(state, dtype=torch.float32).to(DEVICE)
        probs  = self(state)
        dist   = Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action)

env        = gym.make('CartPole-v1', max_episode_steps=2000)
state_dim  = env.observation_space.shape[0]   # 4
action_dim = env.action_space.n               # 2
policy     = PolicyNet(state_dim, action_dim).to(DEVICE)
optimizer  = torch.optim.Adam(policy.parameters(), lr=LR)

n_params = sum(p.numel() for p in policy.parameters())
print(f"Policy network: {n_params} parameters")
print(f"State dim: {state_dim}  |  Action dim: {action_dim}")


## 2. REINFORCE

The REINFORCE update rule is:

$$\nabla_\theta J(\theta) = \mathbb{E}\left[ \sum_t G_t \nabla_\theta \log \pi_\theta(a_t | s_t) \right]$$

where $G_t$ is the **discounted return from timestep $t$**: $G_t = \sum_{k=0}^{T-t} \gamma^k r_{t+k}$.

Actions that led to high future returns get a higher log-probability nudge. The discount factor $\gamma$ means rewards further in the future count less — an intuitive prior that also helps with variance.

**Baseline subtraction:** The mean return from is subtracted from each episode's returns before computing the loss. This doesn't change the expected gradient (the mean is a constant) but dramatically reduces variance, making training much more stable. You can try commenting it out to see the difference.


In [ ]:
def compute_returns(rewards, gamma=GAMMA):
    """Compute discounted returns G_t for each timestep."""
    returns, R = [], 0.0
    for r in reversed(rewards):
        R = r + gamma * R
        returns.insert(0, R)
    returns = torch.tensor(returns, dtype=torch.float32).to(DEVICE)
    # Baseline subtraction: normalise to zero mean, unit variance
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    return returns

def run_episode(env):
    """Run one episode, collect log probs and rewards."""
    state, _  = env.reset(seed=SEED)
    log_probs, rewards = [], []
    for _ in range(RENDER_STEPS):
        action, log_prob = policy.act(state)
        state, reward, terminated, truncated, _ = env.step(action)
        log_probs.append(log_prob)
        rewards.append(reward)
        if terminated or truncated:
            break
    return log_probs, rewards

def update_policy(log_probs, returns):
    """REINFORCE gradient update."""
    loss = -torch.stack(
        [lp * G for lp, G in zip(log_probs, returns)]
    ).sum()
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
    optimizer.step()
    return loss.item()


In [ ]:
def render_episode(env_name='CartPole-v1', max_steps=RENDER_STEPS, label=''):
    """Render one episode of the current policy as an inline HTML video."""
    env_r  = gym.make(env_name, render_mode='rgb_array', max_episode_steps=1000)
    state, _ = env_r.reset(seed=SEED)
    frames   = []
    total_r  = 0
    for _ in range(max_steps):
        frames.append(env_r.render())
        action, _ = policy.act(state)
        state, reward, terminated, truncated, _ = env_r.step(action)
        total_r += reward
        if terminated or truncated:
            break
    env_r.close()

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.axis('off')
    title = ax.set_title(f'{label}  |  steps survived: {len(frames)}', fontsize=11)
    im    = ax.imshow(frames[0])

    def update(frame):
        im.set_data(frames[frame])
        return [im]

    ani = animation.FuncAnimation(fig, update, frames=len(frames),
                                  interval=30, blit=True)
    plt.close(fig)
    return HTML(ani.to_jshtml())


## 4. Training

The training loop runs episodes, computes returns, and updates the policy.  Watch for the episode length to start climbing.  


In [ ]:
env          = gym.make('CartPole-v1', max_episode_steps=2000)
episode_returns = []
recent       = deque(maxlen=50)
videos       = {}

print(f"Training REINFORCE on CartPole-v1 for up to {MAX_EPISODES} episodes...\n")

for ep in range(1, MAX_EPISODES + 1):
    log_probs, rewards = run_episode(env)
    returns            = compute_returns(rewards)
    loss               = update_policy(log_probs, returns)

    total_return = sum(rewards)
    episode_returns.append(total_return)
    recent.append(total_return)
    avg = np.mean(recent)

    if ep % 50 == 0:
        print(f"Episode {ep:4d} | return {total_return:6.1f} | avg50 {avg:6.1f} | loss {loss:7.3f}")

    # Render a video snapshot every RENDER_EVERY episodes
    if ep % RENDER_EVERY == 0 or ep == 1:
        videos[ep] = render_episode(label=f'Episode {ep}')

    if avg >= SOLVED_SCORE and ep >= 50:
        print(f"\nSolved at episode {ep}! avg50 = {avg:.1f}")
        videos[ep] = render_episode(label=f'Episode {ep} — SOLVED')
        break

env.close()
print("Training complete.")


In [ ]:
# --- Return curve ---
plt.figure(figsize=(11, 4))
plt.plot(episode_returns, alpha=0.4, label='Episode return')
# Rolling average
window = 50
roll   = np.convolve(episode_returns, np.ones(window)/window, mode='valid')
plt.plot(range(window-1, len(episode_returns)), roll, linewidth=2, label=f'{window}-ep average')
plt.axhline(SOLVED_SCORE, color='green', linestyle='--', label=f'Solved threshold ({SOLVED_SCORE})')
plt.xlabel('Episode')
plt.ylabel('Total return (steps survived)')
plt.title('REINFORCE on CartPole-v1')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 5. Watching the Agent Improve

Display episodes over the course of training.  The progression from random to controlled should be clear.


In [ ]:
for ep_num, video in sorted(videos.items()):
    print(f"=== Episode {ep_num} ===")
    display(video)


## 6. What's Actually Happening?

A few things worth noting now that you've watched it train:

**Why does it sometimes get worse before it gets better?**
Policy gradient methods have high variance, and a few unlucky episodes can push the policy in the wrong direction.  The 50-episode rolling average smooths this out but individual episode returns will zigzag throughout.  This is the fundamental challenge that actor-critic methods (A2C, PPO) address by adding a learned value function as a baseline.

**What would happen without baseline subtraction?**
Try commenting out the normalisation line in `compute_returns` and rerunning.  Training will still work eventually but will be noticeably noisier and slower.

**Why does this scale badly?**
The state here is 4 hand-crafted numbers.  For a real problem (a robot, a game from pixels) the state space is enormous and we can't just feed it into a small MLP.  That's where Deep Q-Networks (DQN) and proximal policy optimisation (PPO) come in, but the core idea of "make successful actions more probable" is exactly the same.



In [ ]:
# --- Experiment: watch the effect of the discount factor ---
# Retrain with gamma=0.9 and compare the return curve.
# Lower gamma makes the agent more short-sighted — does it still solve CartPole?

GAMMA_EXP = 0.9
policy_exp = PolicyNet(state_dim, action_dim).to(DEVICE)
opt_exp    = torch.optim.Adam(policy_exp.parameters(), lr=LR)

returns_exp = []
env_exp     = gym.make('CartPole-v1', max_episode_steps=2000)

for ep in range(1, 601):
    state, _ = env_exp.reset(seed=SEED)
    log_probs, rewards = [], []
    for _ in range(RENDER_STEPS):
        state_t  = torch.tensor(state, dtype=torch.float32).to(DEVICE)
        probs    = policy_exp(state_t)
        dist     = Categorical(probs)
        action   = dist.sample()
        log_probs.append(dist.log_prob(action))
        state, reward, terminated, truncated, _ = env_exp.step(action.item())
        rewards.append(reward)
        if terminated or truncated:
            break

    # Returns with experimental gamma
    rets, R = [], 0.0
    for r in reversed(rewards):
        R = r + GAMMA_EXP * R
        rets.insert(0, R)
    rets = torch.tensor(rets, dtype=torch.float32).to(DEVICE)
    rets = (rets - rets.mean()) / (rets.std() + 1e-8)

    loss = -torch.stack([lp * G for lp, G in zip(log_probs, rets)]).sum()
    opt_exp.zero_grad()
    loss.backward()
    opt_exp.step()
    returns_exp.append(sum(rewards))

env_exp.close()

# Compare curves
min_len = min(len(episode_returns), len(returns_exp))
plt.figure(figsize=(11, 4))
w = 50
plt.plot(np.convolve(episode_returns[:min_len], np.ones(w)/w, mode='valid'),
         label=f'γ={GAMMA} (original)', linewidth=2)
plt.plot(np.convolve(returns_exp[:min_len],     np.ones(w)/w, mode='valid'),
         label=f'γ={GAMMA_EXP} (short-sighted)', linewidth=2)
plt.axhline(SOLVED_SCORE, color='green', linestyle='--', label='Solved')
plt.xlabel('Episode')
plt.ylabel('50-ep average return')
plt.title('Effect of discount factor γ on REINFORCE')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
